In [0]:
# Notebook: 05_Batch_Scoring
from databricks import feature_store
import mlflow

# --- Configuration ---
model_name = "databricks_us.default.adventureworkssalespredictor"
model_stage = "staging" # Or "Staging" / "Production" if you transitioned it
output_table_name = "adventureworks_db.predictions.sales_predictions2" # Where to save predictions

In [0]:
# --- Load Data to Score ---
# Option 1: Load fresh data from the source database
# Re-use connection logic from 01_Data_Ingestion notebook
# Filter for data that needs scoring (e.g., new orders)

# Option 2: Load data from a Delta table (e.g., daily new orders table)
# scoring_raw_df = spark.read.format("delta").load("/mnt/adventureworks/new_orders_for_scoring")
# --- Connection Configuration ---

jdbc_hostname = dbutils.secrets.get(scope="jdbc-secrets", key="db-host")
jdbc_port = dbutils.secrets.get(scope="jdbc-secrets", key="db-port")
jdbc_database = dbutils.secrets.get(scope="jdbc-secrets", key="db-database")
jdbc_user = dbutils.secrets.get(scope="jdbc-secrets", key="db-user")
jdbc_password = dbutils.secrets.get(scope="jdbc-secrets", key="db-password")

jdbc_url = f"jdbc:postgresql://{jdbc_hostname}:{jdbc_port}/{jdbc_database}"
connection_properties = {
  "user": jdbc_user,
  "password": jdbc_password,
  "driver": "org.postgresql.Driver"
}

# For this example, let's just score the data we originally prepared (for demonstration)
try:
    # We only need the primary keys of the data we want to score
    # scoring_keys_df = spark.read.format("delta").load("/mnt/adventureworks/prepared_data").select("primary_key", "OrderDate") # Add timestamp if needed by features
    # OR reload from DB if not saved

    scoring_keys_df = spark.read.jdbc(
        url=jdbc_url,
        table="Sales.SalesOrderHeader", # Use actual schema.table
        properties=connection_properties
    ).select("SalesOrderID", "OrderDate").withColumnRenamed("SalesOrderID", "primary_key") # Select keys to score

    print(f"Loaded {scoring_keys_df.count()} records to score.")

except Exception as e:
     print(f"Error loading data for scoring: {e}")
     dbutils.notebook.exit("Failed to load scoring data.")


Loaded 31465 records to score.


In [0]:
# --- Score using Feature Store ---
fs = feature_store.FeatureStoreClient()

# Construct the model URI based on name and stage
if model_stage == "None" or not model_stage: # Get latest version if no stage specified
     model_stage = "staging"
     model_uri = f"models:/{model_name}@{model_stage}"
else:
     model_uri = f"models:/{model_name}@{model_stage}"

print(f"Scoring using model: {model_uri}")

Scoring using model: models:/databricks_us.default.[REDACTED]salespredictor@staging


In [0]:
model = mlflow.pyfunc.load_model('models:/databricks_us.default.adventureworkssalespredictor@staging')
model

mlflow.pyfunc.loaded_model:
  artifact_path: model
  flavor: mlflow.sklearn
  run_id: daa05536c2d948d7b54d6de0223ea898

In [0]:
fs.score_batch

<bound method FeatureStoreClient.score_batch of <databricks.feature_store.client.FeatureStoreClient object at 0x7fe70ffa7560>>

In [0]:
from pyspark.sql import functions as F

try:
    # Use score_batch - it automatically looks up features based on the model's Feature Store metadata
    predictions_df = fs.score_batch(
        model_uri='models:/databricks_us.default.adventureworkssalespredictor@staging',
        df=scoring_keys_df, # DataFrame containing primary keys (and timestamp if needed)
        result_type='double' # Or 'string', 'boolean', etc. matching model output type
    )

    print("Batch scoring completed.")
    display(predictions_df.limit(10))

    # --- Save Predictions ---
    # Add timestamp, model version used, etc. for traceability
    predictions_df = predictions_df.withColumn("prediction_timestamp", F.current_timestamp()) \
                                   .withColumn("model_version_used", F.lit(model_uri)) # Store URI or just version

    predictions_df.write.format("delta").mode("overwrite").saveAsTable(output_table_name)
    # Use append mode if adding predictions incrementally
    # predictions_df.write.format("delta").mode("append").saveAsTable(output_table_name)

    print(f"Predictions saved to Delta table: {output_table_name}")
    # dbutils.notebook.exit(output_table_name)

except Exception as e:
    print(f"Error during batch scoring or saving: {e}")
    # dbutils.notebook.exit("Batch scoring failed.")

2025/04/08 12:39:14 WARNING mlflow.pyfunc: Calling `spark_udf()` with `env_manager="local"` does not recreate the same environment that was used during training, which may lead to errors or inaccurate predictions. We recommend specifying `env_manager="conda"`, which automatically recreates the environment that was used to train the model and performs inference in the recreated environment.


2025/04/08 12:39:14 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


Batch scoring completed.


primary_key,OrderDate,CustomerID,SubTotal,TaxAmt,Freight,OrderYear,OrderMonth,OrderDayOfWeek,prediction
43659,2011-05-31T00:00:00Z,29825,20565.621,1971.5149,616.0984,2011,5,3,23141.249512095943
43660,2011-05-31T00:00:00Z,29672,1294.2529,124.2483,38.8276,2011,5,3,1456.609603384046
43661,2011-05-31T00:00:00Z,29734,32726.479,3153.7695,985.553,2011,5,3,36864.81859562047
43662,2011-05-31T00:00:00Z,29994,28832.53,2775.1646,867.2389,2011,5,3,32487.660432310222
43663,2011-05-31T00:00:00Z,29565,419.4589,40.2681,12.5838,2011,5,3,472.06367192157086
43664,2011-05-31T00:00:00Z,29898,24432.61,2344.9922,732.81,2011,5,3,27514.96235016534
43665,2011-05-31T00:00:00Z,29580,14352.771,1375.9427,429.9821,2011,5,3,16126.67962038417
43666,2011-05-31T00:00:00Z,30052,5056.4897,486.3747,151.9921,2011,5,3,5689.660153470917
43667,2011-05-31T00:00:00Z,29974,6107.082,586.1203,183.1626,2011,5,3,6866.242922570426
43668,2011-05-31T00:00:00Z,29614,35944.156,3461.7654,1081.8018,2011,5,3,40516.371810174845


Error during batch scoring or saving: [SCHEMA_NOT_FOUND] The schema `[REDACTED]_db.predictions` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a catalog, verify the current_schema() output, or qualify the name with the correct catalog.
To tolerate the error on drop use DROP SCHEMA IF EXISTS. SQLSTATE: 42704
